# 00 · Colab setup
One-time checks: Drive paths, data layout, output/cache folders, HF token, GPU.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Sexism-research")  # TODO: set to your Drive clone
else:
    PROJECT_ROOT = Path.cwd().parent  # local run from notebooks/
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
HF_HOME = PROJECT_ROOT / "hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

In [ ]:
%pip install -q pandas==3.0.6 numpy==2.5.4 scikit-learn==1.9.1 pyarrow==26.0.0 pyyaml==6.0.3
# src/: schemas, prompts and LLM backends (torch is Colab's build, logged not pinned)
%pip install -q transformers==5.20.1 pydantic==2.13.5 outlines==1.3.3 xgrammar==0.2.8

In [ ]:
# Fail early if the data layout differs from data/CLAUDE.md
EXPECTED = [
    "CMSB/sexism_data.csv",
    "CMSB/sexism_annotations.csv",
    "EDOS/edos_labelled_aggregated.csv",
    "EDOS/edos_labelled_individual_annotations.csv",
    "EXIST/EXIST 2025 Tweets Dataset/training/EXIST2025_training.json",
    "EXIST/EXIST 2025 Tweets Dataset/dev/EXIST2025_dev.json",
    "EXIST/evaluation/golds/EXIST2025_dev_task1_3_gold_hard.json",
    "EXIST/evaluation/golds/EXIST2025_dev_task1_3_gold_soft.json",
]
missing = [p for p in EXPECTED if not (DATA_ROOT / p).exists()]
assert not missing, f"missing data files: {missing}"
for p in EXPECTED:
    print(f"{(DATA_ROOT / p).stat().st_size / 1e6:7.1f} MB  {p}")

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
print("python", sys.version.split()[0])
print("OUTPUT_ROOT", OUTPUT_ROOT)
print("HF_HOME    ", HF_HOME)

In [ ]:
# Token comes from Colab secrets (or the local env), never from a cell
if IN_COLAB:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    from huggingface_hub import whoami

    print("HF user:", whoami(token=HF_TOKEN)["name"])
else:
    print("HF_TOKEN not set: gated models (Llama, Gemma) will fail to download")

In [ ]:
try:
    import torch

    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name}, {props.total_memory / 1e9:.0f} GB, torch {torch.__version__}")
    else:
        print(f"no GPU, torch {torch.__version__}")
except ImportError:
    print("torch not installed")